In [1]:
import awkward as ak
import uproot
import numpy as np
import pandas as pd
import os
import time
import json

from coffea import processor
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

import sys
sys.path.append("..")
from src.processors.Processor import Processor

In [2]:
base = '/stash/user/fudawei/samples/mc/2018'
basedir = {d: os.path.join(base, d) for d in os.listdir(base)}
samples = {s: [] for s in basedir}
EMPTY = False
for s in basedir:
    for (current_path, dirs, files) in os.walk(basedir[s]):
        for f in files:
            if f.endswith('.root'):
                if os.path.getsize(os.path.join(current_path, f)) == 0:
                    EMPTY = True
                    print("Attention! File below is empty!")
                    print(os.path.join(current_path, f))
                samples[s].append(os.path.join(current_path, f))
                
if EMPTY:
    raise Exception("Some files are empty!")

Attention! File below is empty!
/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT2000toInf_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/5FA5C4CA-9DC2-8F42-93C5-125EB8E3BF1C.root
Attention! File below is empty!
/stash/user/fudawei/samples/mc/2018/QCD/QCD_HT100to200_TuneCP5_13TeV-madgraphMLM-pythia8/RunIISummer20UL18NanoAODv9-106X_upgrade2018_realistic_v16_L1v1-v1/NANOAODSIM/A8394F33-04AF-6E46-A0C6-C88B2961B5F3.root


Exception: Some files are empty!

In [4]:
cutflow = {}
t0 = time.time()
                  
def parallelProcess(samples, name):
    global cutflow, t0
    cutflow[name] = processor.run_uproot_job(
        fileset={name: samples[name]},
        treename="Events",
        processor_instance=Processor(outdir=f'../output/{name}'),
        executor=processor.futures_executor,
        executor_args={"schema": NanoAODSchema, "workers": 24}, # running on $workers cpu cores
    )
    cutflow['time_'+name] = (time.time() - t0)/60
    with open('../output/cutflow.txt', 'w') as file:
        json.dump(cutflow, file)

#parallelProcess(samples=samples, name='ZpToHGamma')
parallelProcess(samples=samples, name='GJets')


Output()

concurrent.futures.process._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "/home/fudawei/anaconda3/lib/python3.9/concurrent/futures/process.py", line 246, in _process_worker
    r = call_item.fn(*call_item.args, **call_item.kwargs)
  File "/home/fudawei/anaconda3/lib/python3.9/site-packages/coffea/processor/executor.py", line 1348, in automatic_retries
    raise e
  File "/home/fudawei/anaconda3/lib/python3.9/site-packages/coffea/processor/executor.py", line 1333, in automatic_retries
    return func(*args, **kwargs)
  File "/home/fudawei/anaconda3/lib/python3.9/site-packages/coffea/processor/executor.py", line 1398, in metadata_fetcher
    with uproot.open(item.filename, timeout=xrootdtimeout) as file:
  File "/home/fudawei/anaconda3/lib/python3.9/site-packages/uproot/reading.py", line 141, in open
    file = ReadOnlyFile(
  File "/home/fudawei/anaconda3/lib/python3.9/site-packages/uproot/reading.py", line 580, in __init__
    self._source = Source(
  File "/home/fud

ValueError: cannot mmap an empty file